# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1, df2, df3 = pd.read_csv(url1), pd.read_csv(url2), pd.read_csv(url3)

def clean_columns(dataframe):
    dataframe = dataframe.copy()
    dataframe.columns = dataframe.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
    return dataframe.rename(columns={"st": "state", "custome_lifetime_value": "customer_lifetime_value"})

df1, df2, df3 = clean_columns(df1), clean_columns(df2), clean_columns(df3)

print("File 1 shape:", df1.shape)
print("File 2 shape:", df2.shape)
print("File 3 shape:", df3.shape)

print("\nFile 1 columns:", df1.columns.tolist())
print("\nFile 2 columns:", df2.columns.tolist())
print("\nFile 3 columns:", df3.columns.tolist())

df = pd.concat([df1, df2, df3], ignore_index=True)
df = df.dropna(how="all").reset_index(drop=True)

for column in df.select_dtypes(include=["object", "string"]).columns:
    df[column] = df[column].astype("string").str.strip()

df["gender"] = df["gender"].str.lower().replace({"m": "Male", "male": "Male", "f": "Female", "female": "Female", "femal": "Female"})
df["state"] = df["state"].replace({"AZ": "Arizona", "Cali": "California", "WA": "Washington"})
df["education"] = df["education"].replace({"Bachelors": "Bachelor"})
df["vehicle_class"] = df["vehicle_class"].replace({"Sports Car": "Luxury", "Luxury SUV": "Luxury", "Luxury Car": "Luxury"})

df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"].astype("string").str.replace("%", "", regex=False).str.replace(",", "", regex=False), errors="coerce")

def clean_complaints(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    return pd.to_numeric(value.split("/")[1] if "/" in value and len(value.split("/")) > 1 else value, errors="coerce")

df["number_of_open_complaints"] = df["number_of_open_complaints"].apply(clean_complaints)

for column in ["income", "monthly_premium_auto", "total_claim_amount"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("\nMissing values before cleaning")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_values"))

for column in df.select_dtypes(include=["object", "string"]).columns:
    if not df[column].mode(dropna=True).empty:
        df[column] = df[column].fillna(df[column].mode(dropna=True).iloc[0])

for column in ["customer_lifetime_value", "income", "monthly_premium_auto", "number_of_open_complaints", "total_claim_amount"]:
    df[column] = df[column].fillna(df[column].median())

duplicates_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

df[["customer_lifetime_value", "monthly_premium_auto", "total_claim_amount"]] = df[["customer_lifetime_value", "monthly_premium_auto", "total_claim_amount"]].round(2)
df["income"] = df["income"].round().astype(int)
df["number_of_open_complaints"] = df["number_of_open_complaints"].round().astype(int)

column_order = ["customer", "state", "gender", "education", "customer_lifetime_value", "income", "monthly_premium_auto", "number_of_open_complaints", "policy_type", "vehicle_class", "total_claim_amount"]
df = df[[column for column in column_order if column in df.columns]]

print("\nFinal shape:", df.shape)
print("Duplicates removed:", duplicates_before)
print("Remaining duplicates:", df.duplicated().sum())
print("Remaining missing values:", df.isnull().sum().sum())

display(df.head(10))
display(df.describe().round(2))
display(df.describe(include=["object", "string"]))

df.to_csv("cleaned_customer_data.csv", index=False)
print("\nFile saved as cleaned_customer_data.csv")

File 1 shape: (4008, 11)
File 2 shape: (996, 11)
File 3 shape: (7070, 11)

File 1 columns: ['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount']

File 2 columns: ['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'total_claim_amount', 'policy_type', 'vehicle_class']

File 3 columns: ['customer', 'state', 'customer_lifetime_value', 'education', 'gender', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'total_claim_amount', 'vehicle_class']

Missing values before cleaning


,missing_values
gender,122
customer_lifetime_value,7
customer,0
state,0
education,0
income,0
monthly_premium_auto,0
number_of_open_complaints,0
policy_type,0
vehicle_class,0



Final shape: (9134, 11)
Duplicates removed: 3
Remaining duplicates: 0
Remaining missing values: 0


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,Female,Master,7714.88,0,1000.0,0,Personal Auto,Four-Door Car,2.70
1,QZ44356,Arizona,Female,Bachelor,697953.59,0,94.0,0,Personal Auto,Four-Door Car,1131.46
2,AI49188,Nevada,Female,Bachelor,1288743.17,48767,108.0,0,Personal Auto,Two-Door Car,566.47
3,WW63253,California,Male,Bachelor,764586.18,0,106.0,0,Corporate Auto,SUV,529.88
4,GA49547,Washington,Male,High School or Below,536307.65,36357,68.0,0,Personal Auto,Four-Door Car,17.27
5,OC83172,Oregon,Female,Bachelor,825629.78,62902,69.0,0,Personal Auto,Two-Door Car,159.38
6,XZ87318,Oregon,Female,College,538089.86,55350,67.0,0,Corporate Auto,Four-Door Car,321.60
7,CF85061,Arizona,Male,Master,721610.03,0,101.0,0,Corporate Auto,Four-Door Car,363.03
8,DY87989,Oregon,Male,Bachelor,2412750.4,14072,71.0,0,Corporate Auto,Four-Door Car,511.20
9,BQ94931,Oregon,Female,College,738817.81,28812,93.0,0,Special Auto,Four-Door Car,425.53


,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,total_claim_amount
count,9134.0,9134.00,9134.00,9134.00,9134.00
mean,181937.9,37824.85,110.39,0.38,430.48
std,440903.85,30359.23,581.47,0.91,289.62
min,1898.01,0.00,61.00,0.00,0.10
25%,4650.06,0.00,68.00,0.00,266.96
50%,7714.88,34240.00,83.00,0.00,377.50
75%,26131.72,62446.50,109.00,0.00,546.05
max,5816655.35,99981.00,35354.00,5.00,2893.24


,customer,state,gender,education,policy_type,vehicle_class
count,9134,9134,9134,9134,9134,9134
unique,9056,5,2,5,3,4
top,GA49547,California,Female,Bachelor,Personal Auto,Four-Door Car
freq,2,3150,4726,2742,6790,4640



File saved as cleaned_customer_data.csv


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [2]:
import pandas as pd
from IPython.display import display

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
display(df.head())

sales_channel_revenue = pd.pivot_table(df, index="sales_channel", values="total_claim_amount", aggfunc="sum").rename(columns={"total_claim_amount": "total_revenue"}).sort_values("total_revenue", ascending=False).round(2).reset_index()

display(sales_channel_revenue)

top_channel = sales_channel_revenue.iloc[0]
lowest_channel = sales_channel_revenue.iloc[-1]
revenue_difference = top_channel["total_revenue"] - lowest_channel["total_revenue"]

print(f"The sales channel with the highest total revenue is {top_channel['sales_channel']} with {top_channel['total_revenue']:,.2f}.")
print(f"The sales channel with the lowest total revenue is {lowest_channel['sales_channel']} with {lowest_channel['total_revenue']:,.2f}.")
print(f"The difference between the highest and lowest sales channels is {revenue_difference:,.2f}.")

avg_clv_by_education_gender = pd.pivot_table(df, index="education", columns="gender", values="customer_lifetime_value", aggfunc="mean").round(2)

display(avg_clv_by_education_gender)

clv_segments = avg_clv_by_education_gender.stack().sort_values(ascending=False)

highest_education, highest_gender = clv_segments.index[0]
lowest_education, lowest_gender = clv_segments.index[-1]
highest_value, lowest_value = clv_segments.iloc[0], clv_segments.iloc[-1]

print(f"The segment with the highest average customer lifetime value is {highest_gender} customers with {highest_education} education at {highest_value:,.2f}.")
print(f"The segment with the lowest average customer lifetime value is {lowest_gender} customers with {lowest_education} education at {lowest_value:,.2f}.")

gender_average = df.groupby("gender")["customer_lifetime_value"].mean().round(2).sort_values(ascending=False).to_frame("average_customer_lifetime_value")
display(gender_average)

Dataset shape: (10910, 27)


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


,sales_channel,total_revenue
0,Agent,1810226.82
1,Branch,1301204.00
2,Call Center,926600.82
3,Web,706600.04


The sales channel with the highest total revenue is Agent with 1,810,226.82.
The sales channel with the lowest total revenue is Web with 706,600.04.
The difference between the highest and lowest sales channels is 1,103,626.78.


gender,F,M
education,,
Bachelor,7874.27,7703.60
College,7748.82,8052.46
Doctor,7328.51,7415.33
High School or Below,8675.22,8149.69
Master,8157.05,8168.83


The segment with the highest average customer lifetime value is F customers with High School or Below education at 8,675.22.
The segment with the lowest average customer lifetime value is F customers with Doctor education at 7,328.51.


,average_customer_lifetime_value
gender,
F,8071.11
M,7963.04


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [3]:
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")
df["number_of_open_complaints"] = pd.to_numeric(df["number_of_open_complaints"], errors="coerce").fillna(0)

month_order = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
df["month"] = pd.Categorical(df["effective_to_date"].dt.month_name(), categories=month_order, ordered=True)

complaints_long = df.groupby(["policy_type", "month"], observed=True, as_index=False)["number_of_open_complaints"].sum().rename(columns={"policy_type": "Policy Type", "month": "Month", "number_of_open_complaints": "Number of Complaints"}).sort_values(["Policy Type", "Month"]).reset_index(drop=True)

complaints_long["Number of Complaints"] = complaints_long["Number of Complaints"].round().astype(int)

display(complaints_long)

highest_complaints = complaints_long.loc[complaints_long.groupby("Policy Type")["Number of Complaints"].idxmax()].sort_values("Policy Type").reset_index(drop=True)

print("Month with the highest number of complaints for each policy type")
display(highest_complaints)

overall_highest = complaints_long.loc[complaints_long["Number of Complaints"].idxmax()]

print(f"The highest complaint total was recorded for {overall_highest['Policy Type']} during {overall_highest['Month']}, with {overall_highest['Number of Complaints']} complaints.")

,Policy Type,Month,Number of Complaints
0,Corporate Auto,January,443
1,Corporate Auto,February,385
2,Personal Auto,January,1728
3,Personal Auto,February,1454
4,Special Auto,January,87
5,Special Auto,February,95


Month with the highest number of complaints for each policy type


,Policy Type,Month,Number of Complaints
0,Corporate Auto,January,443
1,Personal Auto,January,1728
2,Special Auto,February,95


The highest complaint total was recorded for Personal Auto during January, with 1728 complaints.
